# LLM Provider Comparison

SynapseKit wraps every LLM provider behind a single interface. The same `agenerate()`, `astream()`, and `LLMConfig` work identically across OpenAI, Anthropic, Groq, Ollama, and 26 other providers. This notebook benchmarks four of the most popular providers on speed, cost, and quality so you can make an informed choice.

**What you'll build:** A benchmark harness that sends the same prompts to OpenAI, Anthropic, Groq, and Ollama, measures latency and token cost, and prints a comparison table.

**Time:** ~15 min | **Difficulty:** Beginner

**What you'll learn:**
- Import and configure providers using `synapsekit.llms.*`
- Use `LLMConfig` to set temperature, max tokens, and other shared parameters
- Call `agenerate()` on any provider with identical code
- Measure latency and estimate cost with `CostTracker`
- Switch providers by changing a single variable

## Prerequisites & Setup

You'll need API keys for the providers you want to benchmark:
- **OpenAI:** https://platform.openai.com/api-keys
- **Anthropic:** https://console.anthropic.com/
- **Groq:** https://console.groq.com/
- **Ollama:** local server — install from https://ollama.ai, then run `ollama serve` and `ollama pull llama3.2`

In [ ]:
!pip install "synapsekit[openai,anthropic,groq,ollama]" -q

In [ ]:
import os

# Set your API keys here, or export them in your shell before launching Jupyter
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["GROQ_API_KEY"] = "gsk_..."
# Ollama: no API key needed — runs locally on port 11434

## Step 1: Import providers and set up shared config

`LLMConfig` is provider-agnostic — the same object works for every provider. Only set fields that the provider supports; unsupported fields are silently ignored.

In [ ]:
import asyncio
import time
from synapsekit import LLMConfig, CostTracker
from synapsekit.llms.openai import OpenAILLM
from synapsekit.llms.anthropic import AnthropicLLM
from synapsekit.llms.groq import GroqLLM
from synapsekit.llms.ollama import OllamaLLM

# LLMConfig is provider-agnostic — the same object works for every provider.
# Only set fields that the provider supports; unsupported fields are silently ignored.
config = LLMConfig(
    temperature=0.2,    # Low temperature for consistent, comparable outputs
    max_tokens=256,     # Cap output so benchmarks are fair
)

print("Config created:", config)

## Step 2: Instantiate each provider

Each provider class accepts a model name and a shared `LLMConfig`. Credentials are read from environment variables automatically.

In [ ]:
# Each provider class accepts a model name and a shared LLMConfig.
# Credentials are read from environment variables automatically.
providers = {
    "openai/gpt-4o-mini":       OpenAILLM(model="gpt-4o-mini",           config=config),
    "anthropic/claude-haiku":   AnthropicLLM(model="claude-haiku-3-5",    config=config),
    "groq/llama-3.1-8b":        GroqLLM(model="llama-3.1-8b-instant",     config=config),
    "ollama/llama3.2":          OllamaLLM(model="llama3.2",               config=config),
}

print(f"Configured {len(providers)} providers:")
for name in providers:
    print(f"  - {name}")

## Step 3: Write the benchmark harness

Run a single prompt across all providers and record latency and response.

In [ ]:
async def benchmark(prompt: str, providers: dict) -> list[dict]:
    """Run a single prompt across all providers and record latency + response."""
    results = []
    tracker = CostTracker()

    for name, llm in providers.items():
        start = time.perf_counter()
        response = await llm.agenerate(prompt)
        elapsed = time.perf_counter() - start

        # Approximate token counts from word count; use tiktoken for precision.
        in_tokens  = len(prompt.split()) * 2
        out_tokens = len(response.text.split())
        rec = tracker.record(name, input_tokens=in_tokens, output_tokens=out_tokens)

        results.append({
            "provider": name,
            "latency_s": round(elapsed, 2),
            "output_tokens": out_tokens,
            "cost_usd": rec.cost_usd,
            "answer": response.text[:120],
        })

    return results

print("benchmark() defined.")

## Step 4: Run the benchmark across multiple prompts

Test prompts span different complexity levels to reveal differences in reasoning quality.

In [ ]:
TEST_PROMPTS = [
    "What is the capital of Australia?",
    "Explain what a hash table is in two sentences.",
    "Write a Python one-liner that reverses a string.",
]

async def run_full_benchmark():
    print(f"{'Provider':<30} {'Latency':>10} {'Tokens':>8} {'Cost USD':>12}")
    print("-" * 65)

    for prompt in TEST_PROMPTS:
        print(f"\nPrompt: {prompt!r}\n")
        results = await benchmark(prompt, providers)

        for r in sorted(results, key=lambda x: x["latency_s"]):
            print(
                f"  {r['provider']:<28} "
                f"{r['latency_s']:>8.2f}s "
                f"{r['output_tokens']:>7} tk "
                f"${r['cost_usd']:>10.7f}"
            )
            print(f"    -> {r['answer']}")

await run_full_benchmark()

## Complete Working Example

A self-contained version wrapped in `asyncio.run(main())` for easy script execution outside Jupyter.

In [ ]:
import asyncio
import time
from synapsekit import LLMConfig, CostTracker
from synapsekit.llms.openai import OpenAILLM
from synapsekit.llms.anthropic import AnthropicLLM
from synapsekit.llms.groq import GroqLLM
from synapsekit.llms.ollama import OllamaLLM

async def main():
    config = LLMConfig(temperature=0.2, max_tokens=256)

    providers = {
        "openai/gpt-4o-mini":     OpenAILLM(model="gpt-4o-mini",        config=config),
        "anthropic/claude-haiku": AnthropicLLM(model="claude-haiku-3-5", config=config),
        "groq/llama-3.1-8b":      GroqLLM(model="llama-3.1-8b-instant",  config=config),
        "ollama/llama3.2":        OllamaLLM(model="llama3.2",            config=config),
    }

    prompt = "What is the difference between a process and a thread?"
    tracker = CostTracker()

    print(f"Prompt: {prompt!r}\n")
    print(f"{'Provider':<30} {'Latency':>10} {'Tokens':>8} {'Cost':>12}")
    print("-" * 65)

    for name, llm in providers.items():
        start = time.perf_counter()
        response = await llm.agenerate(prompt)
        elapsed = time.perf_counter() - start

        in_tok  = len(prompt.split()) * 2
        out_tok = len(response.text.split())
        rec = tracker.record(name, input_tokens=in_tok, output_tokens=out_tok)

        print(
            f"{name:<30} "
            f"{elapsed:>8.2f}s "
            f"{out_tok:>7} tk "
            f"${rec.cost_usd:>10.7f}"
        )
        print(f"  {response.text[:100]}...\n")

    print(f"\nTotal estimated cost: ${tracker.total_cost_usd:.7f}")

asyncio.run(main())

## Variations

In [ ]:
# Switch providers with one variable:
# active_llm = providers["groq/llama-3.1-8b"]
# response = await active_llm.agenerate("Your prompt here")

# Streaming responses:
# async for chunk in llm.astream("Tell me a story"):
#     print(chunk.text, end="", flush=True)

# Run providers concurrently to reduce wall-clock time:
# import asyncio
# responses = await asyncio.gather(*[
#     llm.agenerate(prompt) for llm in providers.values()
# ])

print("Variation examples shown above as comments.")

## Next Steps

- **[Cost-Aware LLM Router](cost-router.ipynb)** — automatically route queries to the cheapest suitable model
- **[LLM Fallback Chains](fallback-chain.ipynb)** — handle provider outages with automatic failover
- **[Semantic Response Caching](semantic-caching.ipynb)** — cache responses to eliminate redundant API calls

**Troubleshooting:**
- `OllamaLLM` raises `ConnectionRefusedError`: Start Ollama with `ollama serve`, then `ollama pull llama3.2`
- `AnthropicLLM` raises `AuthenticationError`: Ensure `ANTHROPIC_API_KEY` starts with `sk-ant-`
- Cost shows `$0.0000000` for Ollama: Expected — Ollama runs locally with no per-token charge